In [13]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Test_Iceberg_Query").getOrCreate()
print("=== DANH SÁCH DATABASE ===")
spark.sql("SHOW DATABASES IN local_catalog").show(truncate=False)

=== DANH SÁCH DATABASE ===
+-----------+
|namespace  |
+-----------+
|default    |
|tiki_bronze|
|tiki_gold  |
|tiki_silver|
+-----------+



In [26]:

spark.sql("SHOW TABLES IN local_catalog.tiki").show(truncate=False)


+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|tiki     |price_history|false      |
|tiki     |products     |false      |
+---------+-------------+-----------+



In [27]:
spark.sql("SHOW TABLES IN local_catalog.tiki_bronze").show(truncate=False)

+-----------+------------+-----------+
|namespace  |tableName   |isTemporary|
+-----------+------------+-----------+
|tiki_bronze|products_raw|false      |
+-----------+------------+-----------+



In [28]:
spark.sql("SHOW TABLES IN local_catalog.tiki_gold").show(truncate=False)

+---------+-----------------+-----------+
|namespace|tableName        |isTemporary|
+---------+-----------------+-----------+
|tiki_gold|brand_performance|false      |
|tiki_gold|price_trend      |false      |
|tiki_gold|discount_analysis|false      |
|tiki_gold|top_products     |false      |
|tiki_gold|daily_summary    |false      |
+---------+-----------------+-----------+



In [4]:
spark.sql("SHOW TABLES IN local_catalog.tiki_silver").show(truncate=False)

+-----------+-------------+-----------+
|namespace  |tableName    |isTemporary|
+-----------+-------------+-----------+
|tiki_silver|price_history|false      |
|tiki_silver|products     |false      |
+-----------+-------------+-----------+



In [24]:
query = """
        describe local_catalog.tiki.products
        
"""
spark.sql(query).show(truncate=False)

+--------------+---------+-------+
|col_name      |data_type|comment|
+--------------+---------+-------+
|id            |bigint   |NULL   |
|sku           |string   |NULL   |
|name          |string   |NULL   |
|url_key       |string   |NULL   |
|price         |bigint   |NULL   |
|original_price|bigint   |NULL   |
|discount      |bigint   |NULL   |
|discount_rate |int      |NULL   |
|brand_name    |string   |NULL   |
|rating_average|double   |NULL   |
|review_count  |int      |NULL   |
|thumbnail_url |string   |NULL   |
|category_id   |int      |NULL   |
|quantity_sold |int      |NULL   |
|crawl_date    |date     |NULL   |
|loaded_at     |timestamp|NULL   |
|source_file   |string   |NULL   |
+--------------+---------+-------+



In [5]:
query = """
        describe local_catalog.tiki_silver.price_history
        
"""
spark.sql(query).show(truncate=False)

+--------------+---------+-------+
|col_name      |data_type|comment|
+--------------+---------+-------+
|id            |bigint   |NULL   |
|sku           |string   |NULL   |
|name          |string   |NULL   |
|url_key       |string   |NULL   |
|price         |bigint   |NULL   |
|original_price|bigint   |NULL   |
|discount      |bigint   |NULL   |
|discount_rate |int      |NULL   |
|brand_name    |string   |NULL   |
|rating_average|double   |NULL   |
|review_count  |int      |NULL   |
|thumbnail_url |string   |NULL   |
|category_id   |int      |NULL   |
|category_name |string   |NULL   |
|quantity_sold |int      |NULL   |
|crawl_date    |date     |NULL   |
|loaded_at     |timestamp|NULL   |
|source_file   |string   |NULL   |
+--------------+---------+-------+



In [20]:
query = """
        SELECT 
            id,
            name,
            crawl_date,
            price AS current_price,
            LAG(price) OVER (PARTITION BY id ORDER BY crawl_date ASC) AS previous_price,
            price - LAG(price) OVER (PARTITION BY id ORDER BY crawl_date ASC) AS price_diff,
            CASE 
                WHEN LAG(price) OVER (PARTITION BY id ORDER BY crawl_date ASC) IS NULL THEN 'Lần đầu ghi nhận'
                WHEN price > LAG(price) OVER (PARTITION BY id ORDER BY crawl_date ASC) THEN 'Tăng giá'
                WHEN price < LAG(price) OVER (PARTITION BY id ORDER BY crawl_date ASC) THEN 'Giảm giá'
                ELSE 'Không đổi'
            END AS price_trend
        FROM local_catalog.tiki_silver.price_history
        ORDER BY id, crawl_date;
        
"""
spark.sql(query).show(50, truncate=False)

+------+-----------------------------------------------------------------------------------------------------------+----------+-------------+--------------+----------+----------------+
|id    |name                                                                                                       |crawl_date|current_price|previous_price|price_diff|price_trend     |
+------+-----------------------------------------------------------------------------------------------------------+----------+-------------+--------------+----------+----------------+
|288994|Son Lì It's Well Plus Lipstick Unlimited Sensual Matte 3.7g - LIP-M3                                       |2026-06-02|89000        |NULL          |NULL      |Lần đầu ghi nhận|
|289016|Son Lì It's Well Plus Lipstick Infinite Supreme Semi Matte 3.7g - LIP-SM3                                  |2026-06-02|89000        |NULL          |NULL      |Lần đầu ghi nhận|
|380290|Sữa Rửa Mặt Dabo Tinh Chất Collagen 180ml                          

In [37]:
query = """
        describe local_catalog.tiki_gold.brand_performance
        
"""
spark.sql(query).show(truncate=False)

+-------------------+---------+-------+
|col_name           |data_type|comment|
+-------------------+---------+-------+
|brand_name         |string   |NULL   |
|total_products     |bigint   |NULL   |
|total_quantity_sold|bigint   |NULL   |
|avg_rating         |double   |NULL   |
|avg_reviews        |double   |NULL   |
|avg_price_k        |double   |NULL   |
|avg_discount_rate  |double   |NULL   |
|last_updated       |date     |NULL   |
+-------------------+---------+-------+



In [38]:
query = """
        select * from local_catalog.tiki_gold.brand_performance
"""
spark.sql(query).show(truncate=False)

+---------------------------+--------------+-------------------+----------+-----------+-----------+-----------------+------------+
|brand_name                 |total_products|total_quantity_sold|avg_rating|avg_reviews|avg_price_k|avg_discount_rate|last_updated|
+---------------------------+--------------+-------------------+----------+-----------+-----------+-----------------+------------+
|Acnes                      |26            |1162993            |4.57      |141.0      |99.5       |18.5             |2026-06-13  |
|Selsun                     |4             |781945             |4.8       |698.0      |96.4       |14.5             |2026-06-13  |
|Nivea                      |61            |494451             |2.23      |2.0        |104.0      |16.8             |2026-06-13  |
|Durex                      |85            |485912             |1.52      |136.0      |277.3      |8.5              |2026-06-13  |
|Nivea Men                  |30            |455421             |2.86      |4.0     

In [19]:
query = """
        WITH history_ranked AS (
            -- Phân loại: Đâu là lần đầu ghi nhận (rn=1), đâu là lần đổi giá (rn>1)
            SELECT 
                id,
                crawl_date,
                ROW_NUMBER() OVER(PARTITION BY id ORDER BY crawl_date ASC) as rn
            FROM local_catalog.tiki_silver.price_history
        ),
        daily_changes AS (
            -- Thống kê số lượng theo từng ngày từ bảng History
            SELECT 
                crawl_date,
                SUM(CASE WHEN rn = 1 THEN 1 ELSE 0 END) as sp_moi_ghi_nhan,
                SUM(CASE WHEN rn > 1 THEN 1 ELSE 0 END) as sp_bien_dong_gia
            FROM history_ranked
            GROUP BY crawl_date
        ),
        daily_total AS (
            -- Lấy tổng số sản phẩm cào được mỗi ngày từ bảng Bronze Raw
            SELECT 
                crawl_date, 
                COUNT(DISTINCT id) as tong_san_pham_crawl
            FROM local_catalog.tiki_bronze.products_raw
            GROUP BY crawl_date
        )
        -- Kết hợp lại để ra báo cáo cuối cùng
        SELECT 
            t.crawl_date,
            t.tong_san_pham_crawl,
            COALESCE(c.sp_moi_ghi_nhan, 0) as sp_moi_ghi_nhan,
            COALESCE(c.sp_bien_dong_gia, 0) as sp_bien_dong_gia,
            (t.tong_san_pham_crawl - COALESCE(c.sp_moi_ghi_nhan, 0) - COALESCE(c.sp_bien_dong_gia, 0)) as sp_khong_doi_gia
        FROM daily_total t
        LEFT JOIN daily_changes c ON t.crawl_date = c.crawl_date
        ORDER BY t.crawl_date DESC;

        """
spark.sql(query).show(truncate=False)

+----------+-------------------+---------------+----------------+----------------+
|crawl_date|tong_san_pham_crawl|sp_moi_ghi_nhan|sp_bien_dong_gia|sp_khong_doi_gia|
+----------+-------------------+---------------+----------------+----------------+
|2026-06-14|18707              |0              |63              |18644           |
|2026-06-13|18659              |40             |170             |18449           |
|2026-06-12|18792              |84             |112             |18596           |
|2026-06-11|18786              |26             |505             |18255           |
|2026-06-10|18803              |83             |120             |18600           |
|2026-06-09|18686              |10             |237             |18439           |
|2026-06-08|18681              |39             |863             |17779           |
|2026-06-07|18685              |13             |790             |17882           |
|2026-06-05|18755              |26             |226             |18503           |
|202

In [9]:
query = """
        select * from local_catalog.tiki_gold.daily_summary
        """
spark.sql(query).show(truncate=False)

+----------+--------------+------------+----------------+-----------+-----------------+-------------------+-------------------+
|crawl_date|total_products|total_brands|total_categories|avg_price_k|avg_discount_rate|total_quantity_sold|flash_sale_products|
+----------+--------------+------------+----------------+-----------+-----------------+-------------------+-------------------+
|2026-06-13|18660         |2146        |240             |1036.5     |5.4              |5644633            |700                |
|2026-06-12|18792         |2166        |240             |1032.8     |5.3              |5923880            |701                |
|2026-06-11|18786         |2164        |240             |1032.6     |5.2              |5947215            |682                |
+----------+--------------+------------+----------------+-----------+-----------------+-------------------+-------------------+



In [12]:
spark.sql("""
    SELECT count(id) as so_sp_bien_dong, crawl_date 
    FROM local_catalog.tiki_silver.price_history 
    GROUP BY crawl_date
    Order by crawl_date DESC
""").show()

+---------------+----------+
|so_sp_bien_dong|crawl_date|
+---------------+----------+
|             71|2026-06-14|
|          18840|2026-06-13|
|            109|2026-06-12|
|            504|2026-06-11|
+---------------+----------+



In [19]:
spark.sql("""
    SELECT SUM(CASE WHEN discount_rate >= 50 THEN 1 ELSE 0 END) AS flash_sale_products
    FROM local_catalog.tiki_bronze.products_raw
    where crawl_date = date(now())
""").show()

+-------------------+
|flash_sale_products|
+-------------------+
|                700|
+-------------------+



In [10]:
query = """
        select * from local_catalog.tiki_gold.price_trend
"""
spark.sql(query).show(1000, truncate=False)

+----------+-----------+-------------------------------------+-------------------+---------------+---------------+---------------+---------------------+
|crawl_date|category_id|category_name                        |total_price_changes|avg_new_price_k|min_new_price_k|max_new_price_k|avg_new_discount_rate|
+----------+-----------+-------------------------------------+-------------------+---------------+---------------+---------------+---------------------+
|2026-06-13|1583       |Sữa rửa mặt                          |638                |379.6          |15.0           |6800.0         |6.8                  |
|2026-06-13|1596       |Phấn nền, kem nền                    |63                 |500.8          |45.0           |2062.5         |10.6                 |
|2026-06-13|1597       |Phấn phủ                             |64                 |472.1          |77.0           |1480.0         |9.9                  |
|2026-06-13|1598       |Sữa tắm                              |266                |

In [34]:
# 1. Xóa các bảng bên trong trước
spark.sql("DROP TABLE IF EXISTS local_catalog.tiki.products")
spark.sql("DROP TABLE IF EXISTS local_catalog.tiki.price_history")

# 2. Xóa Database tiki cũ
spark.sql("DROP NAMESPACE IF EXISTS local_catalog.tiki")

# Delete Database in Data Lakehouse

DataFrame[]

In [29]:
# 1. Tạo namespace mới (nếu chưa có)
spark.sql("CREATE NAMESPACE IF NOT EXISTS local_catalog.tiki_silver")

# 2. Copy toàn bộ dữ liệu (cả cấu trúc lẫn data) từ bảng cũ sang bảng mới
spark.sql("""
    CREATE TABLE local_catalog.tiki_silver.products 
    USING iceberg 
    AS SELECT * FROM local_catalog.tiki.products
""")

spark.sql("""
    CREATE TABLE local_catalog.tiki_silver.price_history 
    USING iceberg 
    AS SELECT * FROM local_catalog.tiki.price_history
""")

# Data Migration of Data Engineering

DataFrame[]

In [6]:
print("=== LỊCH SỬ CÁC LẦN GHI DỮ LIỆU ===")
spark.sql("SELECT * FROM local_catalog.tiki.products.history").show(truncate=False)

print("=== CHI TIẾT SNAPSHOTS ===")
spark.sql("SELECT committed_at, snapshot_id, operation FROM local_catalog.tiki.products.snapshots").show(truncate=False)


# Time Travel in Data Lakehouse

=== LỊCH SỬ CÁC LẦN GHI DỮ LIỆU ===
+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-05-25 07:41:15.762|212357408956329652 |NULL               |true               |
|2026-05-25 07:42:18.757|8761820206006598383|212357408956329652 |true               |
|2026-05-25 08:49:59.182|4698979343288364922|8761820206006598383|true               |
|2026-05-27 12:28:40.798|8241903772937420669|4698979343288364922|true               |
|2026-05-27 12:31:11.661|6755189955897460993|8241903772937420669|true               |
|2026-05-27 12:54:51.992|8407684545818752076|6755189955897460993|true               |
|2026-05-28 09:06:49.559|9135974879012586705|8407684545818752076|true               |
+-----------------------+-------------------+-------------------+-------------------+

=== CHI TIẾT SNAP